In [ ]:
# %% [markdown]
# # 12 — Plan Recommendation Engine
#
# ## Objective
# Build the first plan recommendation engine for new prospects.
#
# This notebook will:
# - combine prospect profile outputs with existing policy structure
# - define plan fit rules
# - rank plans for each prospect
# - generate recommendation outputs
#
# Main output:
# - `prospect_plan_recommendations.csv`

# %%
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

# %%
import pandas as pd
import numpy as np

from src.config import OUTPUTS_DIR
from src.data.load_data import load_policies

# %% [markdown]
# ## Load data

# %%
prospect_profile_scores = pd.read_csv(OUTPUTS_DIR / "prospect_profile_scores.csv")
policies = load_policies()

print("prospect_profile_scores:", prospect_profile_scores.shape)
print("policies:", policies.shape)

display(prospect_profile_scores.head(3))
display(policies.head(3))

# %% [markdown]
# ## Helper functions

# %%
def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def safe_num(x):
    return pd.to_numeric(x, errors="coerce")

# %% [markdown]
# ## Detect prospect columns

# %%
id_col = first_existing(prospect_profile_scores, ["prospect_id", "id"])
price_col = first_existing(prospect_profile_scores, ["price_sensitivity", "price_preference", "budget_sensitivity"])
coverage_col = first_existing(prospect_profile_scores, ["coverage_preference", "coverage_priority"])
network_col = first_existing(prospect_profile_scores, ["network_preference", "network_priority"])
plan_pref_col = first_existing(prospect_profile_scores, ["preferred_plan_type", "plan_preference"])

detected_cols = {
    "id_col": id_col,
    "price_col": price_col,
    "coverage_col": coverage_col,
    "network_col": network_col,
    "plan_pref_col": plan_pref_col,
}

pd.DataFrame(detected_cols.items(), columns=["field", "detected_column"])

# %% [markdown]
# ## Build plan catalog
#
# We create a catalog of unique plan configurations from policies.csv.

# %%
plan_catalog_cols = [c for c in [
    "plan_type",
    "plan_tier",
    "coverage_scope",
    "provider_network_type",
    "deductible_amount",
    "copay_level",
    "annual_coverage_limit",
    "maternity_coverage_flag",
    "pharmacy_coverage_flag",
    "chronic_care_program_flag",
    "premium_monthly",
    "premium_annual",
    "recommended_plan_flag",
] if c in policies.columns]

plan_catalog = (
    policies[plan_catalog_cols]
    .drop_duplicates()
    .reset_index(drop=True)
)

for col in ["deductible_amount", "annual_coverage_limit", "premium_monthly", "premium_annual"]:
    if col in plan_catalog.columns:
        plan_catalog[col] = pd.to_numeric(plan_catalog[col], errors="coerce")

print("plan_catalog:", plan_catalog.shape)
display(plan_catalog.head(10))

# %% [markdown]
# ## Enrich plan catalog with portfolio medians by plan_type / plan_tier

# %%
group_cols = [c for c in ["plan_type", "plan_tier"] if c in policies.columns]

if len(group_cols) > 0:
    enrich_cols = [c for c in ["premium_monthly", "premium_annual", "deductible_amount", "annual_coverage_limit"] if c in policies.columns]
    plan_enrichment = (
        policies[group_cols + enrich_cols]
        .copy()
    )

    for col in enrich_cols:
        plan_enrichment[col] = pd.to_numeric(plan_enrichment[col], errors="coerce")

    plan_enrichment = (
        plan_enrichment.groupby(group_cols)[enrich_cols]
        .median()
        .reset_index()
        .rename(columns={col: f"{col}_median" for col in enrich_cols})
    )

    plan_catalog = plan_catalog.merge(plan_enrichment, on=group_cols, how="left")

display(plan_catalog.head(10))

# %% [markdown]
# ## Plan fit scoring logic

# %%
catalog_premium_median = safe_num(plan_catalog["premium_monthly"]).median() if "premium_monthly" in plan_catalog.columns else np.nan
catalog_deductible_median = safe_num(plan_catalog["deductible_amount"]).median() if "deductible_amount" in plan_catalog.columns else np.nan
catalog_coverage_limit_median = safe_num(plan_catalog["annual_coverage_limit"]).median() if "annual_coverage_limit" in plan_catalog.columns else np.nan

print("catalog_premium_median:", catalog_premium_median)
print("catalog_deductible_median:", catalog_deductible_median)
print("catalog_coverage_limit_median:", catalog_coverage_limit_median)

# %%
def score_plan_fit(prospect_row, plan_row):
    score = 0.0
    reasons = []

    risk_score = safe_num(prospect_row.get("prospect_risk_score"))
    est_mid = safe_num(prospect_row.get("estimated_monthly_premium_mid"))

    price_pref = str(prospect_row.get(price_col, "")).strip().lower() if price_col is not None else ""
    coverage_pref = str(prospect_row.get(coverage_col, "")).strip().lower() if coverage_col is not None else ""
    network_pref = str(prospect_row.get(network_col, "")).strip().lower() if network_col is not None else ""
    plan_pref = str(prospect_row.get(plan_pref_col, "")).strip().lower() if plan_pref_col is not None else ""

    plan_type = str(plan_row.get("plan_type", "")).strip().lower()
    plan_tier = str(plan_row.get("plan_tier", "")).strip().lower()
    coverage_scope = str(plan_row.get("coverage_scope", "")).strip().lower()
    network_type = str(plan_row.get("provider_network_type", "")).strip().lower()

    premium_monthly = safe_num(plan_row.get("premium_monthly"))
    deductible = safe_num(plan_row.get("deductible_amount"))
    coverage_limit = safe_num(plan_row.get("annual_coverage_limit"))
    pharmacy_flag = safe_num(plan_row.get("pharmacy_coverage_flag"))
    chronic_flag = safe_num(plan_row.get("chronic_care_program_flag"))
    recommended_flag = safe_num(plan_row.get("recommended_plan_flag"))

    # 1. Risk vs coverage richness
    if pd.notna(risk_score):
        if risk_score >= 0.75:
            if ("premium" in plan_tier) or ("gold" in plan_tier) or ("high" in coverage_scope) or ("full" in coverage_scope):
                score += 3
                reasons.append("high_risk_good_coverage_fit")

            if pd.notna(deductible) and pd.notna(catalog_deductible_median) and deductible <= catalog_deductible_median:
                score += 1
                reasons.append("high_risk_lower_deductible_fit")

            if pd.notna(coverage_limit) and pd.notna(catalog_coverage_limit_median) and coverage_limit >= catalog_coverage_limit_median:
                score += 1
                reasons.append("high_risk_higher_coverage_limit_fit")

            if pd.notna(chronic_flag) and chronic_flag == 1:
                score += 1
                reasons.append("high_risk_chronic_program_fit")

        elif risk_score <= 0.35:
            if ("basic" in plan_tier) or ("standard" in plan_tier) or ("limited" in coverage_scope) or ("essential" in coverage_scope):
                score += 2
                reasons.append("low_risk_efficient_coverage_fit")

            if pd.notna(deductible) and pd.notna(catalog_deductible_median) and deductible >= catalog_deductible_median:
                score += 1
                reasons.append("low_risk_higher_deductible_fit")

        else:
            if ("standard" in plan_tier) or ("mid" in coverage_scope) or ("balanced" in plan_type):
                score += 2
                reasons.append("medium_risk_balanced_fit")

    # 2. Price fit
    if pd.notna(est_mid) and pd.notna(premium_monthly):
        rel_gap = abs(premium_monthly - est_mid) / max(est_mid, 1)
        if rel_gap <= 0.15:
            score += 3
            reasons.append("price_band_close_fit")
        elif rel_gap <= 0.30:
            score += 2
            reasons.append("price_band_reasonable_fit")
        elif rel_gap <= 0.50:
            score += 1
            reasons.append("price_band_tolerable_fit")

    # 3. Explicit coverage preference
    if coverage_pref:
        if any(k in coverage_pref for k in ["high", "broad", "full", "premium", "wide"]):
            if any(k in coverage_scope for k in ["high", "full", "premium", "broad", "wide"]):
                score += 2
                reasons.append("coverage_preference_match")
        elif any(k in coverage_pref for k in ["basic", "essential", "low", "limited"]):
            if any(k in coverage_scope for k in ["basic", "essential", "low", "limited"]):
                score += 2
                reasons.append("coverage_preference_match")

    # 4. Network preference
    if network_pref:
        if network_pref in network_type or network_type in network_pref:
            score += 1.5
            reasons.append("network_preference_match")

    # 5. Preferred plan type
    if plan_pref:
        if plan_pref in plan_type or plan_type in plan_pref:
            score += 2
            reasons.append("plan_type_preference_match")

    # 6. Price sensitivity
    if price_pref and pd.notna(premium_monthly) and pd.notna(catalog_premium_median):
        if any(k in price_pref for k in ["high", "sensitive", "budget", "low_cost", "cheap"]):
            if premium_monthly <= catalog_premium_median:
                score += 2
                reasons.append("price_sensitive_fit")
        elif any(k in price_pref for k in ["low", "not_sensitive", "flexible"]):
            score += 0.5

    # 7. Pharmacy and chronic extras
    if pd.notna(pharmacy_flag) and pharmacy_flag == 1 and pd.notna(risk_score) and risk_score >= 0.50:
        score += 0.5
        reasons.append("pharmacy_fit")

    # 8. Portfolio prior recommendation marker
    if pd.notna(recommended_flag) and recommended_flag == 1:
        score += 0.5
        reasons.append("historical_recommended_marker")

    return round(score, 3), "; ".join(reasons) if reasons else "general_fit"

# %% [markdown]
# ## Score every plan for every prospect

# %%
recommendation_rows = []

for _, prospect_row in prospect_profile_scores.iterrows():
    for _, plan_row in plan_catalog.iterrows():
        fit_score, fit_reason = score_plan_fit(prospect_row, plan_row)

        recommendation_rows.append({
            "prospect_id": prospect_row[id_col],
            "prospect_risk_score": prospect_row.get("prospect_risk_score", np.nan),
            "prospect_risk_segment": prospect_row.get("prospect_risk_segment", np.nan),
            "predicted_archetype": prospect_row.get("predicted_archetype", np.nan),
            "estimated_monthly_premium_low": prospect_row.get("estimated_monthly_premium_low", np.nan),
            "estimated_monthly_premium_mid": prospect_row.get("estimated_monthly_premium_mid", np.nan),
            "estimated_monthly_premium_high": prospect_row.get("estimated_monthly_premium_high", np.nan),
            "plan_type": plan_row.get("plan_type", np.nan),
            "plan_tier": plan_row.get("plan_tier", np.nan),
            "coverage_scope": plan_row.get("coverage_scope", np.nan),
            "provider_network_type": plan_row.get("provider_network_type", np.nan),
            "deductible_amount": plan_row.get("deductible_amount", np.nan),
            "copay_level": plan_row.get("copay_level", np.nan),
            "annual_coverage_limit": plan_row.get("annual_coverage_limit", np.nan),
            "premium_monthly": plan_row.get("premium_monthly", np.nan),
            "premium_annual": plan_row.get("premium_annual", np.nan),
            "recommended_plan_flag": plan_row.get("recommended_plan_flag", np.nan),
            "plan_fit_score": fit_score,
            "plan_fit_reason": fit_reason,
        })

recommendation_long = pd.DataFrame(recommendation_rows)
print("recommendation_long:", recommendation_long.shape)
display(recommendation_long.head(10))

# %% [markdown]
# ## Rank plans by prospect

# %%
recommendation_long = recommendation_long.sort_values(
    ["prospect_id", "plan_fit_score", "premium_monthly"],
    ascending=[True, False, True]
).reset_index(drop=True)

recommendation_long["rank_within_prospect"] = (
    recommendation_long.groupby("prospect_id").cumcount() + 1
)

display(recommendation_long.head(20))

# %% [markdown]
# ## Build top recommendations table

# %%
top_n = recommendation_long[recommendation_long["rank_within_prospect"] <= 3].copy()

best_plan = top_n[top_n["rank_within_prospect"] == 1].copy()
best_plan = best_plan.rename(columns={
    "plan_type": "recommended_plan_type",
    "plan_tier": "recommended_plan_tier",
    "coverage_scope": "recommended_coverage_scope",
    "provider_network_type": "recommended_provider_network_type",
    "deductible_amount": "recommended_deductible_amount",
    "copay_level": "recommended_copay_level",
    "annual_coverage_limit": "recommended_annual_coverage_limit",
    "premium_monthly": "recommended_premium_monthly",
    "premium_annual": "recommended_premium_annual",
    "plan_fit_score": "recommended_plan_fit_score",
    "plan_fit_reason": "recommended_plan_fit_reason",
})

recommendations = best_plan[[
    "prospect_id",
    "prospect_risk_score",
    "prospect_risk_segment",
    "predicted_archetype",
    "estimated_monthly_premium_low",
    "estimated_monthly_premium_mid",
    "estimated_monthly_premium_high",
    "recommended_plan_type",
    "recommended_plan_tier",
    "recommended_coverage_scope",
    "recommended_provider_network_type",
    "recommended_deductible_amount",
    "recommended_copay_level",
    "recommended_annual_coverage_limit",
    "recommended_premium_monthly",
    "recommended_premium_annual",
    "recommended_plan_fit_score",
    "recommended_plan_fit_reason",
]].copy()

alt1 = top_n[top_n["rank_within_prospect"] == 2][[
    "prospect_id", "plan_type", "plan_tier", "premium_monthly", "plan_fit_score"
]].rename(columns={
    "plan_type": "alternative_1_plan_type",
    "plan_tier": "alternative_1_plan_tier",
    "premium_monthly": "alternative_1_premium_monthly",
    "plan_fit_score": "alternative_1_plan_fit_score",
})

alt2 = top_n[top_n["rank_within_prospect"] == 3][[
    "prospect_id", "plan_type", "plan_tier", "premium_monthly", "plan_fit_score"
]].rename(columns={
    "plan_type": "alternative_2_plan_type",
    "plan_tier": "alternative_2_plan_tier",
    "premium_monthly": "alternative_2_premium_monthly",
    "plan_fit_score": "alternative_2_plan_fit_score",
})

recommendations = recommendations.merge(alt1, on="prospect_id", how="left")
recommendations = recommendations.merge(alt2, on="prospect_id", how="left")

print("recommendations:", recommendations.shape)
display(recommendations.head(10))

# %% [markdown]
# ## Add explanation text

# %%
def build_explanation(row):
    parts = []

    if pd.notna(row.get("prospect_risk_segment")):
        parts.append(f"Risk segment: {row['prospect_risk_segment']}")

    if pd.notna(row.get("predicted_archetype")):
        parts.append(f"Profile archetype: {row['predicted_archetype']}")

    if pd.notna(row.get("recommended_plan_tier")):
        parts.append(f"Recommended tier: {row['recommended_plan_tier']}")

    if pd.notna(row.get("recommended_premium_monthly")):
        parts.append(f"Estimated monthly premium: {round(row['recommended_premium_monthly'], 2)}")

    if pd.notna(row.get("recommended_plan_fit_reason")):
        parts.append(f"Main fit drivers: {row['recommended_plan_fit_reason']}")

    return " | ".join(parts)

recommendations["recommendation_explanation"] = recommendations.apply(build_explanation, axis=1)

display(recommendations[["prospect_id", "recommendation_explanation"]].head(10))

# %% [markdown]
# ## Save output

# %%
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUTS_DIR / "prospect_plan_recommendations.csv"
recommendations.to_csv(output_file, index=False)

print("Saved:", output_file)

# %% [markdown]
# ## Final notes

# %%
notes = [
    "This is the first plan recommendation engine of the project.",
    "It is designed to be explainable and demo-friendly rather than actuarially final.",
    "These outputs are ready to feed the Prospect Profiler dashboard page."
]

for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

PROJECT_ROOT: C:\Users\dolivares\Desktop\novares-securehealth
prospect_profile_scores: (7000, 16)
policies: (4000, 25)


,prospect_id,age,sex,dependents_n,smoker_flag,bmi_group,chronic_condition_flag,network_preference,risk_points,prospect_risk_score,prospect_risk_segment,predicted_archetype,expected_utilization_proxy,estimated_monthly_premium_low,estimated_monthly_premium_mid,estimated_monthly_premium_high
0,PRS000001,30,M,1,0,Normal,0,Broad,0,0.000000,very_low,balanced_standard,0.069355,258.339280,287.043645,315.748010
1,PRS000002,47,F,1,0,Normal,0,Narrow,1,0.142857,very_low,balanced_standard,0.168203,307.546762,341.718625,375.890488
2,PRS000003,18,F,1,0,Overweight,0,Balanced,1,0.142857,very_low,balanced_standard,0.121429,307.546762,341.718625,375.890488


,policy_id,member_id,policy_start_date,policy_end_date,active_flag,plan_type,plan_tier,coverage_scope,provider_network_type,deductible_amount,...,premium_monthly,premium_annual,underwriting_load_factor,discount_factor,recommended_plan_flag,pricing_adequacy_ratio,renewal_flag,cancellation_flag,sales_channel,broker_id
0,POL000001,MBR000001,2022-09-24,2026-12-31,1,Managed Review,Mid,Full,Balanced,199.02,...,291.4,3496.8,1.139,0.092,1,0.851,1,0,Corporate,NaN
1,POL000002,MBR000002,2024-04-02,2026-12-31,1,Essential,Basic,Ambulatory,Narrow,597.15,...,198.7,2384.4,1.138,0.082,1,1.285,1,0,Direct,NaN
2,POL000003,MBR000003,2024-06-27,2026-12-31,1,Chronic Care,Premium,Full,Broad,196.63,...,430.1,5161.2,1.005,0.060,1,0.994,0,0,Direct,NaN


plan_catalog: (4000, 13)


,plan_type,plan_tier,coverage_scope,provider_network_type,deductible_amount,copay_level,annual_coverage_limit,maternity_coverage_flag,pharmacy_coverage_flag,chronic_care_program_flag,premium_monthly,premium_annual,recommended_plan_flag
0,Managed Review,Mid,Full,Balanced,199.02,Medium,30000,0,1,0,291.40,3496.80,1
1,Essential,Basic,Ambulatory,Narrow,597.15,High,15000,1,0,0,198.70,2384.40,1
2,Chronic Care,Premium,Full,Broad,196.63,Low,55000,0,1,1,430.10,5161.20,1
3,Essential,Basic,Ambulatory,Balanced,452.99,High,15000,1,0,0,159.46,1913.52,1
4,Chronic Care,Premium,Full,Balanced,216.26,Low,55000,0,1,1,515.13,6181.56,1
5,Chronic Care,Premium,Full,Broad,151.76,Low,55000,0,1,1,637.99,7655.88,1
6,Standard,Mid,Full,Balanced,435.73,Medium,35000,0,1,0,342.34,4108.08,1
7,Standard,Mid,Full,Balanced,382.97,Medium,35000,1,1,0,275.21,3302.52,1
8,Chronic Care,Premium,Full,Broad,110.43,Low,55000,0,1,1,704.61,8455.32,1
9,Standard,Mid,Full,Balanced,289.49,Medium,35000,0,1,0,308.27,3699.24,1


,plan_type,plan_tier,coverage_scope,provider_network_type,deductible_amount,copay_level,annual_coverage_limit,maternity_coverage_flag,pharmacy_coverage_flag,chronic_care_program_flag,premium_monthly,premium_annual,recommended_plan_flag,premium_monthly_median,premium_annual_median,deductible_amount_median,annual_coverage_limit_median
0,Managed Review,Mid,Full,Balanced,199.02,Medium,30000,0,1,0,291.40,3496.80,1,380.030,4560.36,247.210,30000.0
1,Essential,Basic,Ambulatory,Narrow,597.15,High,15000,1,0,0,198.70,2384.40,1,171.405,2056.86,426.225,15000.0
2,Chronic Care,Premium,Full,Broad,196.63,Low,55000,0,1,1,430.10,5161.20,1,590.475,7085.70,154.025,55000.0
3,Essential,Basic,Ambulatory,Balanced,452.99,High,15000,1,0,0,159.46,1913.52,1,171.405,2056.86,426.225,15000.0
4,Chronic Care,Premium,Full,Balanced,216.26,Low,55000,0,1,1,515.13,6181.56,1,590.475,7085.70,154.025,55000.0
5,Chronic Care,Premium,Full,Broad,151.76,Low,55000,0,1,1,637.99,7655.88,1,590.475,7085.70,154.025,55000.0
6,Standard,Mid,Full,Balanced,435.73,Medium,35000,0,1,0,342.34,4108.08,1,301.320,3615.84,313.170,35000.0
7,Standard,Mid,Full,Balanced,382.97,Medium,35000,1,1,0,275.21,3302.52,1,301.320,3615.84,313.170,35000.0
8,Chronic Care,Premium,Full,Broad,110.43,Low,55000,0,1,1,704.61,8455.32,1,590.475,7085.70,154.025,55000.0
9,Standard,Mid,Full,Balanced,289.49,Medium,35000,0,1,0,308.27,3699.24,1,301.320,3615.84,313.170,35000.0


catalog_premium_median: 354.515
catalog_deductible_median: 264.155
catalog_coverage_limit_median: 35000.0
